# Flower Bagging - Analisi Risultati
## NECSTLab - Polimi LS2

Analisi del modello federato appena addestrato con Flower Bagging

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from pathlib import Path
import sys

sys.path.append('..')
from utils import DataLoader

# Setup plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Carica Modello Federato

In [ ]:
# Carica modello finale
model_path = Path('../benchmarks/flower_bagging/results/final_model.json')

if model_path.exists():
    bst = xgb.Booster()
    bst.load_model(str(model_path))
    print(f"✅ Modello caricato da: {model_path}")
    print(f"   Numero di alberi: {bst.num_boosted_rounds()}")
else:
    print(f"❌ Modello non trovato: {model_path}")

## 2. Metriche di Training per Round

Analisi della convergenza durante i 10 round FL

In [ ]:
# Metriche estratte dall'output di Flower
mae_per_round = {
    1: 11.37,
    2: 10.89,
    3: 11.17,
    4: 11.63,
    5: 11.49,
    6: 11.72,
    7: 11.66,
    8: 11.93,
    9: 11.99,
    10: 11.88
}

# Crea DataFrame
df_training = pd.DataFrame({
    'Round': list(mae_per_round.keys()),
    'MAE': list(mae_per_round.values())
})

print("📊 Statistiche MAE:")
print(f"   Migliore: {df_training['MAE'].min():.2f} (Round {df_training.loc[df_training['MAE'].idxmin(), 'Round']:.0f})")
print(f"   Peggiore: {df_training['MAE'].max():.2f} (Round {df_training.loc[df_training['MAE'].idxmax(), 'Round']:.0f})")
print(f"   Media: {df_training['MAE'].mean():.2f}")
print(f"   Std Dev: {df_training['MAE'].std():.2f}")

display(df_training)

In [ ]:
# Plot convergenza
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df_training['Round'], df_training['MAE'], 
        marker='o', linewidth=2, markersize=8, label='MAE')
ax.axhline(y=df_training['MAE'].min(), color='green', 
           linestyle='--', alpha=0.7, label=f'Best MAE: {df_training["MAE"].min():.2f}')
ax.axhline(y=df_training['MAE'].mean(), color='orange', 
           linestyle='--', alpha=0.7, label=f'Mean MAE: {df_training["MAE"].mean():.2f}')

ax.set_xlabel('Round FL', fontsize=12)
ax.set_ylabel('MAE (Mean Absolute Error)', fontsize=12)
ax.set_title('Convergenza Flower Bagging - 10 Rounds', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, 11))

plt.tight_layout()
plt.show()

## 3. Valutazione su Test Set

Testa il modello finale su dati di test centralizzati

In [ ]:
# Carica test set centralizzato
test_csv = Path('../data/x_test.csv')

if test_csv.exists():
    df_test = pd.read_csv(test_csv)
    print(f"✅ Test set caricato: {len(df_test)} esempi")
    
    # Separa features e target
    if 'label' in df_test.columns:
        X_test = df_test.drop('label', axis=1)
        y_test = df_test['label']
        
        # Crea DMatrix
        dtest = xgb.DMatrix(X_test, label=y_test)
        
        # Predizione
        y_pred = bst.predict(dtest)
        
        # Calcola metriche
        from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
        
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        print(f"\n📈 Performance su Test Set Centralizzato:")
        print(f"   MAE:  {mae:.2f}")
        print(f"   RMSE: {rmse:.2f}")
        print(f"   R²:   {r2:.3f}")
    else:
        print("⚠️ Colonna 'label' non trovata nel test set")
else:
    print(f"⚠️ Test set non trovato: {test_csv}")

In [ ]:
# Visualizza predizioni vs valori reali
if 'y_pred' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scatter plot: Predizioni vs Reali
    axes[0].scatter(y_test, y_pred, alpha=0.5, s=30)
    axes[0].plot([y_test.min(), y_test.max()], 
                 [y_test.min(), y_test.max()], 
                 'r--', lw=2, label='Perfect Prediction')
    axes[0].set_xlabel('Sleep Quality - Reale', fontsize=11)
    axes[0].set_ylabel('Sleep Quality - Predetta', fontsize=11)
    axes[0].set_title('Predizioni vs Valori Reali', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Distribuzione errori
    errors = y_test - y_pred
    axes[1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
    axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    axes[1].set_xlabel('Errore di Predizione', fontsize=11)
    axes[1].set_ylabel('Frequenza', fontsize=11)
    axes[1].set_title('Distribuzione Errori', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 4. Feature Importance

Analizza quali feature contribuiscono maggiormente alle predizioni

In [ ]:
# Estrai feature importance
importance = bst.get_score(importance_type='gain')

if importance:
    # Ordina per importanza
    importance_df = pd.DataFrame({
        'Feature': list(importance.keys()),
        'Gain': list(importance.values())
    }).sort_values('Gain', ascending=False)
    
    print(f"📊 Top 15 Feature più Importanti:")
    display(importance_df.head(15))
    
    # Plot top 20
    fig, ax = plt.subplots(figsize=(10, 8))
    top_20 = importance_df.head(20)
    
    ax.barh(range(len(top_20)), top_20['Gain'])
    ax.set_yticks(range(len(top_20)))
    ax.set_yticklabels(top_20['Feature'])
    ax.invert_yaxis()
    ax.set_xlabel('Gain', fontsize=11)
    ax.set_title('Top 20 Feature Importance (by Gain)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Nessuna feature importance disponibile")

## 5. Statistiche Modello

In [ ]:
# Statistiche generali
print("🔍 Informazioni Modello Federato:")
print(f"   Numero totale alberi: {bst.num_boosted_rounds()}")
print(f"   Tempo training: 202.71 secondi (~3.4 minuti)")
print(f"   Numero client: 9")
print(f"   Numero round FL: 10")
print(f"   Epoch locali per client: 5")
print(f"   Alberi per client per round: 5")
print(f"   Totale alberi aggregati: {9 * 10 * 5}")

## 6. Conclusioni

**Risultati Chiave:**

1. **Convergenza**: Il modello ha raggiunto il MAE migliore (10.89) al round 2, con una certa variabilità nei round successivi
2. **Performance Finale**: MAE ~11.88 sul validation set federato
3. **Architettura**: Bagging di alberi XGBoost addestrati in modo distribuito su 9 client
4. **Efficienza**: ~3.4 minuti per 10 round completi con simulazione locale

**Prossimi Passi:**
- Confrontare con approccio Cyclic
- Confrontare con NVIDIA FLARE
- Ottimizzare iperparametri (learning rate, max_depth)
- Testare con più round per valutare convergenza